## Anlysis of IMG/PR dataset of plasmids

An environmental database containing 693k plasmid sequences automatically identified from isolate genomes, metagenomes, metatranscriptomes, and environmental samples.

Paper: https://pubmed.ncbi.nlm.nih.gov/37930866/ 
HF data subset (metadata + sequences): https://huggingface.co/datasets/HuggingFaceTB/IMGPR-10k-subset

In [31]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datasets import load_dataset
import numpy as np

# Set plotly template for nice plots
import plotly.io as pio
pio.templates.default = "plotly_white"

In [ ]:
print("Loading IMG/PR dataset (10l subset)...")
dataset = load_dataset(
    "HuggingFaceTB/IMGPR-10k-subset",
    split="train"
)

print(f"Dataset loaded: {len(dataset):,} sequences")
print(f"\nColumns: {dataset.column_names}")

Loading IMG/PR dataset...
Dataset loaded: 10,000 sequences

Columns: ['plasmid_id', 'ptu', 'taxon_oid', 'scaffold_oid', 'source_type', 'ecosystem', 'length', 'gene_count', 'genomad_score', 'putatively_complete', 'topology', 'mob_genes', 't4cp_genes', 't4ss_atpase_genes', 'other_conjugation_genes', 'complete_mpf_family', 'origin_of_transfer', 'arg_genes', 'putative_phage_plasmid', 'host_prediction_method', 'host_taxonomy', 'closest_reference', 'closest_reference_ani_percent', 'closest_reference_af_percent', 'id_sequence', 'description', 'sequence', 'length_sequence']


In [3]:
# Convert to pandas for easier analysis
df = dataset.to_pandas()
print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (10000, 28)


,plasmid_id,ptu,taxon_oid,scaffold_oid,source_type,ecosystem,length,gene_count,genomad_score,putatively_complete,...,putative_phage_plasmid,host_prediction_method,host_taxonomy,closest_reference,closest_reference_ani_percent,closest_reference_af_percent,id_sequence,description,sequence,length_sequence
0,IMGPR_plasmid_3300029368_000048,PTU_00009091,3300029368,Ga0243967_101182,Metagenome,Host-associated;Mammals: Human;Digestive syste...,12866,19,0.9808,No,...,No,CRISPR spacer match,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,None,NaN,NaN,IMGPR_plasmid_3300029368_000048|3300029368|Ga0...,IMGPR_plasmid_3300029368_000048|3300029368|Ga0...,ACACCTGTAAGAGAGATACTTACCGATTGCGATTACAAAAATGTCA...,12866
1,IMGPR_plasmid_3300020816_000030,PTU_00005715,3300020816,Ga0214090_11411060,Metagenome,Engineered;Solid waste;Food waste,5985,7,0.9949,No,...,No,None,None,NZ_CP022726.1,98.65,100.00,IMGPR_plasmid_3300020816_000030|3300020816|Ga0...,IMGPR_plasmid_3300020816_000030|3300020816|Ga0...,GGTACCGGCGAGTGACAGATAACCCTCAGGGCGAAAATCCGGTAGC...,5985
2,IMGPR_plasmid_3300034069_000827,PTU_00001092,3300034069,Ga0370205_002689,Metagenome,Host-associated;Plants;Phyllosphere;Carposphere,21485,26,0.9852,No,...,No,CRISPR spacer match,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,NZ_CP063118.1,99.91,100.00,IMGPR_plasmid_3300034069_000827|3300034069|Ga0...,IMGPR_plasmid_3300034069_000827|3300034069|Ga0...,TTAGCACAAATGAGGGGGTAACGGTATGCTAAACCATTAAGAATTT...,21485
3,IMGPR_plasmid_3300014596_000021,PTU_00223452,3300014596,Ga0121469_100149,Metagenome,Engineered;Built environment;City;Subway;Metal...,10444,12,0.9904,No,...,No,None,None,NZ_CP032113.1,99.98,80.62,IMGPR_plasmid_3300014596_000021|3300014596|Ga0...,IMGPR_plasmid_3300014596_000021|3300014596|Ga0...,TTCAGGGGTATATTTTAATTTTGTCATCGGGATAGTCTCTCAGAAT...,10444
4,IMGPR_plasmid_2795385959_000001,PTU_00000845,2795385959,2795448231,Isolate,None,25076,31,0.9945,Yes,...,No,Isolate taxonomy,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,NZ_CP065267.1,99.10,100.00,IMGPR_plasmid_2795385959_000001|2795385959|279...,IMGPR_plasmid_2795385959_000001|2795385959|279...,GTTGAACGGATTGGCTGTCAGCTGCGCACACAGCGCACGGACCTGG...,25076


## Basic statistics

In [8]:
# Print basic statistics
print("\n=== BASIC STATISTICS ===")
print(f"Total sequences: {len(df):,}")
print(f"\ngeNomad Score:")
print(df['genomad_score'].describe())
print(f"\nLength:")
print(df['length'].describe())
print(f"\nGene Count:")
print(df['gene_count'].describe())
print(f"\nSource Type:")
print(df['source_type'].describe())


=== BASIC STATISTICS ===
Total sequences: 10,000

geNomad Score:
count    10000.000000
mean         0.975640
std          0.029084
min          0.280300
25%          0.970600
50%          0.985800
75%          0.992200
max          0.996600
Name: genomad_score, dtype: float64

Length:
count    1.000000e+04
mean     2.385820e+04
std      5.344486e+04
min      1.568000e+03
25%      5.679750e+03
50%      1.007050e+04
75%      2.245800e+04
max      1.999547e+06
Name: length, dtype: float64

Gene Count:
count    10000.000000
mean        25.570600
std         51.107348
min          0.000000
25%          8.000000
50%         12.000000
75%         24.000000
max       1761.000000
Name: gene_count, dtype: float64

Source Type:
count          10000
unique             5
top       Metagenome
freq            7877
Name: source_type, dtype: object


## geNomad Score Distribution

This shows the confidence level that sequences are actually plasmids. Higher scores = more confident.

In [7]:
fig = px.histogram(
    df,
    x='genomad_score',
    nbins=100,
    title='geNomad Score Distribution (Plasmid Confidence)',
    labels={'genomad_score': 'geNomad Score', 'count': 'Number of Sequences'},
    color_discrete_sequence=['#636EFA']
)

# Add vertical lines for reference thresholds
fig.add_vline(x=0.7, line_dash="dash", line_color="orange", 
              annotation_text="Default threshold (0.7)", annotation_position="top left")
fig.add_vline(x=0.85, line_dash="dash", line_color="red", 
              annotation_text="High confidence (0.85)", annotation_position="top right")

fig.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig.show()

# Print statistics
print(f"Sequences with score >= 0.70: {(df['genomad_score'] >= 0.70).sum():,} ({(df['genomad_score'] >= 0.70).mean()*100:.1f}%)")
print(f"Sequences with score >= 0.85: {(df['genomad_score'] >= 0.85).sum():,} ({(df['genomad_score'] >= 0.85).mean()*100:.1f}%)")
print(f"Sequences with score >= 0.90: {(df['genomad_score'] >= 0.90).sum():,} ({(df['genomad_score'] >= 0.90).mean()*100:.1f}%)")
below_70 = (df['genomad_score'] < 0.70).mean() * 100
below_90 = (df['genomad_score'] < 0.90).mean() * 100
print(f"Percentage of samples below 70%: {below_70:.1f}%")
print(f"Percentage of samples below 90%: {below_90:.1f}%")


Sequences with score >= 0.70: 9,994 (99.9%)
Sequences with score >= 0.85: 9,915 (99.2%)
Sequences with score >= 0.90: 9,720 (97.2%)
Percentage of samples below 70%: 0.1%
Percentage of samples below 90%: 2.8%


#### 💡 Filtering idea: remove the 2.8% samples with a geNomad score below 90%

## Source Type Distribution

Shows whether sequences come from isolate genomes (high quality) or metagenomes (lower quality, often fragmented).

In [15]:
# Count source types
source_counts = df['source_type'].value_counts().reset_index()
source_counts.columns = ['source_type', 'count']
source_counts['percentage'] = (source_counts['count'] / len(df) * 100).round(1)

fig = px.bar(
    source_counts,
    x='source_type',
    y='count',
    title='Source Type Distribution',
    labels={'source_type': 'Source Type', 'count': 'Number of Sequences'},
    text='count',
    color='source_type',
    color_discrete_sequence=px.colors.qualitative.Set2
)


fig.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig.show()

print("\nSource Type Breakdown:")
print(source_counts)


Source Type Breakdown:
                   source_type  count  percentage
0                   Metagenome   7877        78.8
1                      Isolate   2013        20.1
2            Metatranscriptome     74         0.7
3  Metagenome-assembled genome     31         0.3
4      Single amplified genome      5         0.0


#### 💡 Question: are the niche sources (Metatranscriptome, Simple amplified genome.. really needed) if not we can remove them

## Sequence length Distribution

Shows the size distribution of plasmid sequences. Typical plasmids are 2-200 kb.

In [32]:
# Create histogram with linear scale only
fig = go.Figure()

# Original plot (no cap)
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=df['length'], nbinsx=100, name='Length', marker_color='#636EFA')
)

fig.update_xaxes(title_text="Length (bp)")
fig.update_yaxes(title_text="Count")

fig.update_layout(
    height=500,
    showlegend=False,
    title_text="Plasmid Length Distribution (Linear Scale)",
    font=dict(size=12)
)

fig.show()

# Now: plot capped at 200,000 bp below
lengths_capped = df['length'].clip(upper=200_000)

fig_cap = go.Figure()
fig_cap.add_trace(
    go.Histogram(
        x=lengths_capped, 
        nbinsx=100, 
        name='Length (≤200k)', 
        marker_color='#EF553B'
    )
)

fig_cap.update_xaxes(title_text="Length (bp, capped at 200k)", range=[0, 200_000])
fig_cap.update_yaxes(title_text="Count")
fig_cap.update_layout(
    height=500,
    showlegend=False,
    title_text="Plasmid Length Distribution (Capped at 200k, Linear Scale)",
    font=dict(size=12)
)
fig_cap.show()


# Print length statistics
print(f"\nLength Statistics:")
print(f"Min: {df['length'].min():,} bp")
print(f"Max: {df['length'].max():,} bp")
print(f"Mean: {df['length'].mean():,.0f} bp")
print(f"Median: {df['length'].median():,.0f} bp")
print(f"\nSequences < 2 kb: {(df['length'] < 2000).sum():,} ({(df['length'] < 2000).mean()*100:.1f}%)")
print(f"Sequences 2-10 kb: {((df['length'] >= 2000) & (df['length'] < 10000)).sum():,} ({((df['length'] >= 2000) & (df['length'] < 10000)).mean()*100:.1f}%)")
print(f"Sequences 10-50 kb: {((df['length'] >= 10000) & (df['length'] < 50000)).sum():,} ({((df['length'] >= 10000) & (df['length'] < 50000)).mean()*100:.1f}%)")
print(f"Sequences 50-100 kb: {((df['length'] >= 50000) & (df['length'] < 100000)).sum():,} ({((df['length'] >= 50000) & (df['length'] < 100000)).mean()*100:.1f}%)")
print(f"Sequences 100-200 kb: {((df['length'] >= 100000) & (df['length'] < 200000)).sum():,} ({((df['length'] >= 100000) & (df['length'] < 200000)).mean()*100:.1f}%)")
print(f"Sequences > 200 kb: {(df['length'] >= 200000).sum():,} ({(df['length'] >= 200000).mean()*100:.1f}%)")


Length Statistics:
Min: 1,568 bp
Max: 1,999,547 bp
Mean: 23,858 bp
Median: 10,070 bp

Sequences < 2 kb: 5 (0.1%)
Sequences 2-10 kb: 4,973 (49.7%)
Sequences 10-50 kb: 4,019 (40.2%)
Sequences 50-100 kb: 621 (6.2%)
Sequences 100-200 kb: 258 (2.6%)
Sequences > 200 kb: 124 (1.2%)


#### 💡 Note: A small proportion of samples are extremely large (2.6% exceed 100kb and 6.2% exceed 50kb), but the 50kb+ samples account for about half of all tokens 
=> Ask domain experts to review these very large samples to determine whether their quality is good. We found that removing them drives the loss down

## Putatively Complete Distribution

Shows whether sequences are predicted to be complete or fragments.

In [23]:
# Count completeness
complete_counts = df['putatively_complete'].value_counts().reset_index()
complete_counts.columns = ['putatively_complete', 'count']
complete_counts['percentage'] = (complete_counts['count'] / len(df) * 100).round(1)

fig = px.bar(
    complete_counts,
    x='putatively_complete',
    y='count',
    title='Putatively Complete Distribution',
    labels={'putatively_complete': 'Putatively Complete', 'count': 'Number of Sequences'},
    text='count',
    color='putatively_complete',
    color_discrete_map={'Yes': '#00CC96', 'No': '#EF553B'}
)

# fig.update_traces(texttemplate='%{text:,}<br>(%{customdata[0]}%)', textposition='outside')
# fig.update_traces(customdata=complete_counts[['percentage']].values)

fig.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig.show()

print("\nCompleteness Breakdown:")
print(complete_counts)


Completeness Breakdown:
  putatively_complete  count  percentage
0                  No   7771        77.7
1                 Yes   2229        22.3


## Gene Count Distribution

In [25]:
fig = px.histogram(
    df,
    x='gene_count',
    nbins=100,
    title='Gene Count Distribution',
    labels={'gene_count': 'Number of Genes', 'count': 'Number of Sequences'},
    color_discrete_sequence=['#AB63FA']
)

fig.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig.show()

# Add an additional plot capped at 200 genes
fig_capped = px.histogram(
    df[df['gene_count'] <= 200],
    x='gene_count',
    nbins=50,
    title='Gene Count Distribution (Capped at 200 Genes)',
    labels={'gene_count': 'Number of Genes (≤ 200)', 'count': 'Number of Sequences'},
    color_discrete_sequence=['#19D3F3']
)

fig_capped.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig_capped.show()

print(f"\nGene Count Statistics:")
print(f"Min: {df['gene_count'].min()}")
print(f"Max: {df['gene_count'].max()}")
print(f"Mean: {df['gene_count'].mean():.1f}")
print(f"Median: {df['gene_count'].median():.0f}")
print(f"\nSequences with 0 genes: {(df['gene_count'] == 0).sum():,} ({(df['gene_count'] == 0).mean()*100:.1f}%)")
print(f"Sequences with 1-5 genes: {((df['gene_count'] >= 1) & (df['gene_count'] <= 5)).sum():,} ({((df['gene_count'] >= 1) & (df['gene_count'] <= 5)).mean()*100:.1f}%)")


Gene Count Statistics:
Min: 0
Max: 1761
Mean: 25.6
Median: 12

Sequences with 0 genes: 1 (0.0%)
Sequences with 1-5 genes: 1,111 (11.1%)


## Length vs Gene Count (Gene Density)

This scatter plot shows the relationship between plasmid length and number of genes. Plasmids typically have high gene density.

In [26]:
# Sample for plotting if dataset is too large
plot_df = df.sample(n=min(50000, len(df)), random_state=42)

# Calculate gene density (genes per kb)
plot_df['gene_density'] = plot_df['gene_count'] / (plot_df['length'] / 1000)

fig = px.scatter(
    plot_df,
    x='length',
    y='gene_count',
    color='gene_density',
    title=f'Length vs Gene Count (n={len(plot_df):,} sampled sequences)',
    labels={'length': 'Length (bp)', 'gene_count': 'Gene Count', 'gene_density': 'Gene Density<br>(genes/kb)'},
    opacity=0.5,
    color_continuous_scale='Viridis',
    hover_data=['gene_density']
)

fig.update_layout(
    height=600,
    font=dict(size=12)
)

fig.show()

# Print gene density statistics
df['gene_density'] = df['gene_count'] / (df['length'] / 1000)
print(f"\nGene Density Statistics (genes per kb):")
print(f"Mean: {df['gene_density'].mean():.2f}")
print(f"Median: {df['gene_density'].median():.2f}")
print(f"Typical bacterial gene density is ~1 gene/kb")


Gene Density Statistics (genes per kb):
Mean: 1.21
Median: 1.17
Typical bacterial gene density is ~1 gene/kb


Clear correlation

## Completeness by Source Type

Shows whether isolate genomes have more complete plasmids than metagenomes.

In [27]:
# Create crosstab
crosstab = pd.crosstab(df['source_type'], df['putatively_complete'], normalize='index') * 100
crosstab = crosstab.round(1)

# Get counts for labels
counts = pd.crosstab(df['source_type'], df['putatively_complete'])

fig = go.Figure()

for col in crosstab.columns:
    fig.add_trace(go.Bar(
        name=col,
        x=crosstab.index,
        y=crosstab[col],
        text=[f"{pct:.1f}%<br>({cnt:,})" for pct, cnt in zip(crosstab[col], counts[col])],
        textposition='inside'
    ))

fig.update_layout(
    title='Completeness by Source Type',
    xaxis_title='Source Type',
    yaxis_title='Percentage',
    barmode='stack',
    height=500,
    font=dict(size=12),
    legend_title_text='Putatively Complete'
)

fig.show()

print("\nCompleteness by Source Type:")
print(counts)
print("\nPercentages:")
print(crosstab)


Completeness by Source Type:
putatively_complete            No   Yes
source_type                            
Isolate                      1647   366
Metagenome                   6020  1857
Metagenome-assembled genome    28     3
Metatranscriptome              72     2
Single amplified genome         4     1

Percentages:
putatively_complete            No   Yes
source_type                            
Isolate                      81.8  18.2
Metagenome                   76.4  23.6
Metagenome-assembled genome  90.3   9.7
Metatranscriptome            97.3   2.7
Single amplified genome      80.0  20.0


## Topology Distribution

In [29]:
# Count topologies
topology_counts = df['topology'].value_counts().reset_index()
topology_counts.columns = ['topology', 'count']
topology_counts['percentage'] = (topology_counts['count'] / len(df) * 100).round(1)

fig = px.bar(
    topology_counts,
    x='topology',
    y='count',
    title='Topology Distribution',
    labels={'topology': 'Topology', 'count': 'Number of Sequences'},
    text='count',
    color='topology',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# fig.update_traces(texttemplate='%{text:,}<br>(%{customdata[0]}%)', textposition='outside')
fig.update_traces(customdata=topology_counts[['percentage']].values)

fig.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig.show()

print("\nTopology Breakdown:")
print(topology_counts)


Topology Breakdown:
                   topology  count  percentage
0                    Linear   7846        78.5
1    Direct terminal repeat   2053        20.5
2                Concatemer     55         0.5
3  Inverted terminal repeat     46         0.5


#### 💡 Question: what are these categories?

## Putative Phage-Plasmid Distribution

In [30]:
# Count phage-plasmids
phage_plasmid_counts = df['putative_phage_plasmid'].value_counts().reset_index()
phage_plasmid_counts.columns = ['putative_phage_plasmid', 'count']
phage_plasmid_counts['percentage'] = (phage_plasmid_counts['count'] / len(df) * 100).round(1)

fig = px.bar(
    phage_plasmid_counts,
    x='putative_phage_plasmid',
    y='count',
    title='Putative Phage-Plasmid Distribution',
    labels={'putative_phage_plasmid': 'Putative Phage-Plasmid', 'count': 'Number of Sequences'},
    text='count',
    color='putative_phage_plasmid',
    color_discrete_map={'Yes': '#FFA15A', 'No': '#19D3F3'}
)

# fig.update_traces(texttemplate='%{text:,}<br>(%{customdata[0]}%)', textposition='outside')
fig.update_traces(customdata=phage_plasmid_counts[['percentage']].values)

fig.update_layout(
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig.show()

print("\nPhage-Plasmid Breakdown:")
print(phage_plasmid_counts)
print("\nNote: Phage-plasmids are ambiguous hybrids that should likely be filtered out.")


Phage-Plasmid Breakdown:
  putative_phage_plasmid  count  percentage
0                     No   9936        99.4
1                    Yes     64         0.6

Note: Phage-plasmids are ambiguous hybrids that should likely be filtered out.


#### 💡 Filtering idea: remove the 0.6% samples with putative_phage_plasmid=Yes

## Summary: filtering

Let's see how many sequences remain after applying quality filters.

In [ ]:
print("=== FILTERING SUMMARY ===")
print(f"\nTotal sequences: {len(df):,}")

# Apply various filters
filters = {
    'genomad_score >= 0.85': df['genomad_score'] >= 0.85,
    'genomad_score >= 0.90': df['genomad_score'] >= 0.90,
    'source_type == Isolate': df['source_type'] == 'Isolate',
    'putatively_complete == Yes': df['putatively_complete'] == 'Yes',
    'length >= 2000': df['length'] >= 2000,
    'length >= 5000': df['length'] >= 5000,
    'gene_count >= 1': df['gene_count'] >= 1,
    'putative_phage_plasmid == No': df['putative_phage_plasmid'] == 'No',
}

for filter_name, filter_mask in filters.items():
    count = filter_mask.sum()
    pct = (count / len(df) * 100)
    print(f"{filter_name:40s}: {count:8,} ({pct:5.1f}%)")

# Combined high-quality filter
print("\n" + "="*60)
print("RECOMMENDED HIGH-QUALITY SUBSET:")
print("="*60)

high_quality = (
    (df['genomad_score'] >= 0.85) &
    (df['source_type'] == 'Isolate') &
    (df['length'] >= 2000) &
    (df['gene_count'] >= 1) &
    (df['putative_phage_plasmid'] == 'No')
)

print(f"\nFilters applied:")
print(f"  - genomad_score >= 0.85")
print(f"  - source_type == 'Isolate'")
print(f"  - length >= 2000")
print(f"  - gene_count >= 1")
print(f"  - putative_phage_plasmid == 'No'")
print(f"\nHigh-quality sequences: {high_quality.sum():,} ({high_quality.mean()*100:.1f}%)")

# Even stricter filter
very_high_quality = (
    (df['genomad_score'] >= 0.90) &
    (df['source_type'] == 'Isolate') &
    (df['putatively_complete'] == 'Yes') &
    (df['length'] >= 5000) &
    (df['gene_count'] >= 3) &
    (df['putative_phage_plasmid'] == 'No')
)

print(f"\n" + "="*60)
print("VERY HIGH-QUALITY SUBSET (stricter):")
print("="*60)
print(f"\nFilters applied:")
print(f"  - genomad_score >= 0.90")
print(f"  - source_type == 'Isolate'")
print(f"  - putatively_complete == 'Yes'")
print(f"  - length >= 5000")
print(f"  - gene_count >= 3")
print(f"  - putative_phage_plasmid == 'No'")
print(f"\nVery high-quality sequences: {very_high_quality.sum():,} ({very_high_quality.mean()*100:.1f}%)")

In [34]:

dataset_filtered = dataset.filter(lambda x: (x['genomad_score'] >= 0.9) &
    ((x['source_type'] == 'Isolate') | (x['source_type'] == 'Metagenome')) &
    (x['length'] <= 50000) &
    (x['gene_count'] >= 1) &
    (x['putative_phage_plasmid'] == 'No'))

print(f"Number of sequences in filtered dataset: {len(dataset_filtered):,} | Removed: {len(dataset) - len(dataset_filtered):,} ({100 * (len(dataset) - len(dataset_filtered)) / len(dataset):.1f}% of original)")

Number of sequences in filtered dataset: 8,582 | Removed: 1,418 (14.2% of original)
